# 01 — Analyse exploratoire (EDA)

**Problème :** classification binaire — détecter les transactions frauduleuses (`Class` = 1) parmi les transactions légitimes (`Class` = 0).

**Contenu de ce notebook :** chargement des données depuis `data/`, statistiques descriptives, valeurs manquantes, visualisations (distribution des classes, corrélations, distributions de variables clés). Les figures sont enregistrées dans `outputs/`.

In [ ]:
# Racine du projet sur sys.path (exécuter ce notebook depuis notebooks/ ou ajuster)
import sys
from pathlib import Path

_root = Path.cwd().resolve()
if _root.name == "notebooks":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.paths import DATA_PATH, OUTPUT_DIR
from src.eda_plotting import (
    missing_value_report,
    plot_class_distribution,
    plot_correlation_heatmap,
    plot_feature_distributions,
    setup_plot_style,
)

import pandas as pd

setup_plot_style()
TARGET = "Class"
print("Jeu de données :", DATA_PATH.resolve())
print("Sorties figures :", OUTPUT_DIR.resolve())
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Placez creditcard.csv dans data/ ou exécutez data/download_creditcard.py (instructions)."
    )

In [ ]:
# Chargement et aperçu structurel
df = pd.read_csv(DATA_PATH)
if TARGET in df.columns:
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = df.dropna(subset=[TARGET]).reset_index(drop=True)
    df[TARGET] = df[TARGET].astype(int)

print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
display(df.head())
df.info()
display(df.describe())

In [ ]:
# Qualité des données et graphiques EDA
miss = missing_value_report(df)
print("Valeurs manquantes :")
display(miss[miss["count"] > 0] if miss["count"].any() else miss.head())
print("Total cellules manquantes :", int(miss["count"].sum()))

plot_class_distribution(df, TARGET, OUTPUT_DIR / "class_distribution.png")
plot_correlation_heatmap(df, OUTPUT_DIR / "correlation_heatmap.png", exclude=[TARGET])

dist_cols = ["Time", "Amount", "V12", "V14", "V17"]
plot_feature_distributions(
    df, dist_cols, TARGET, OUTPUT_DIR / "feature_distributions.png"
)

**Synthèse EDA :** le jeu est fortement déséquilibré ; les corrélations entre `V*` reflètent la structure PCA. Le prétraitement, le rééchantillonnage SMOTE et la modélisation sont traités dans `02_Modeling.ipynb`.